### Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup the temperature range (-15 to -1°C)
T_surface_C = np.linspace(-15, -1, 100)

# 2. Define dependent variables based on your constraints
# Base is 1°C colder than surface
T_base_C = T_surface_C - 1.0 
# Realistic dew point (slightly lower than surface temp to stay below saturation)
T_dew_C = T_surface_C - 2.0 

# 3. Calculate Densities (rho_calc)
# D2 / Jonas Diss
rho_D2 = 650.0 * np.exp(0.277 * T_surface_C)

# D8
rho_D8 = 207.0 * np.exp(0.266 * T_surface_C - 0.0615 * T_base_C)

# Da Silva Paper
a, b, c = 494.0, 0.11, -0.06
rho_da_silva = a * np.exp(b * T_surface_C + c * T_dew_C)

# 4. Plotting
plt.figure(figsize=(10, 6))
plt.plot(T_surface_C, rho_D2, label='D2 (Jonas Diss)', linewidth=2)
plt.plot(T_surface_C, rho_D8, label='D8', linewidth=2)
plt.plot(T_surface_C, rho_da_silva, label='Da Silva Paper', linewidth=2, linestyle='--')

plt.title('Comparison of Density Correlations')
plt.xlabel('Surface Temperature (°C)')
plt.ylabel('Density ($kg/m^3$)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

print(rho_D2)

In [ ]:
from pathlib import Path
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from frost_evaporator import MultiExperimentAnalyzer, FrostEvaporatorSimulation, SimulationVisualizer


##############################################################################################
# Opti Abt
##############################################################################################

path_exp_abt = Path(r"D:\mbc_nba\OptiAbt_Daten\Versuche\CSV_Export")
experiment_groups_abt = {
    # "+3 °C":  [52, 53, 55, 56, 67],
    "+0 °C":  [40, 41, 42, 43, 44],
    "-7 °C":  [27, 28, 29, 30],
    "-12 °C": [13, 18],


    # ALTERNATIVES
    # "+0 °C (Version 1)":  [3, 4, 5, 7, 8],
    # "+0 °C (Version 2)":  [9, 10, 11, 16, 20],
    # "+0 °C (Version 3)":  [20, 22, 23, 25, 36],
    # "+0 °C (Version 4)":  [45, 46, 47, 48, 49],

    # "-12 °C (Full Version)": [13, 18, 26],


    
}


# experiment_groups_abt = {
#     "Abt-a":  [40],
#     "Abt-b":  [28],
#     "Abt-c":  [13],
# }

# experiment_groups_abt = {
#     "Abt-a": [56],
#     "Abt-b": [40],
#     "Abt-c": [28],
#     "Abt-d": [13],
# }



analyzer_abt = MultiExperimentAnalyzer(experiment_type='OptiAbt')
sim_abt = FrostEvaporatorSimulation("config_OptiAbt.yaml")
viz_abt = SimulationVisualizer(sim_abt.params)

cutoff_pct = 0.05


for group_name, experiment_ids in experiment_groups_abt.items():
    
    print(f"\n{'='*40}")
    print(f"Processing Group: {group_name}")
    print(f"IDs: {experiment_ids}")
    print(f"{'='*40}")

    try:
        # Run analysis
        exp_data = analyzer_abt.analyze(
            exp_ids=experiment_ids, 
            data_path=path_exp_abt, 
            cutoff_pct=cutoff_pct, 
            time_step=sim_abt.params.time_step
        )

        # Run Simulation
        states_history, inputs_history = sim_abt.run_validation(exp_data)

        # Plot 
        viz_abt.plot_comparison(states_history, inputs_history, experiment_ids, path_exp_abt, cutoff_pct, save_fig=False, group_name=group_name, experiment_name="OptiAbt")
        # print(viz_abt.get_relative_error_table(states_history, inputs_history, experiment_ids, path_exp_abt, cutoff_pct, experiment_name="OptiAbt"))
        # viz_abt.plot_layer_temperatures(states_history)
        # viz_abt.plot_detailed(states_history, inputs_history, experiment_ids)
        # viz_abt.plot_roughness(states_history, experiment_ids[0])
        # viz_abt.plot_frost_distribution(states_history)
        # viz_abt.plot_debug(states_history, inputs_history, experiment_ids[0])

    except Exception as e:
        import traceback
        print(f"!!! CRITICAL FAILURE in Group {group_name} !!!")
        print(f"Error Message: {e}")
        traceback.print_exc()

###############################################################################################
# Opti Horst
###############################################################################################

# path_exp_horst = Path(r"D:\mbc_nba\OptiHorst\Daten\Abtauen\CSV_Export")
# experiment_groups_horst = {
#     "+6°C":  [96,97,119,120,121],
#     "-1°C":  [25,26,27,64,28],
#     "-3°C":  [44,45,46,75],
#     "-5°C":  [78,102,103,104],
#     "-10°C": [60,61,62,83,84],

#     # # Alternatives
#     # "+6°C (Version 2)":  [12,13,15,71],
#     # "+6°C (Version 3)":  [87,89,92,100],
#     # "-1°C (Version 2)":  [65,29,30,31],
#     # "-3°C (Version 2)":  [34,35,36,63],
#     # "-5°C (Version 2)":  [56,77,115],
#     # "-10°C (Version 2)": [51,52],


# }


# # experiment_groups_horst = {
# #     "Horst-a": [96],
# #     "Horst-b": [25],
# #     "Horst-c": [45],
# #     "Horst-d": [102],
# #     "Horst-e": [60],
# # }




# analyzer_horst = MultiExperimentAnalyzer(experiment_type='OptiHorst')
# sim_horst      = FrostEvaporatorSimulation("config_OptiHorst.yaml")
# viz_horst      = SimulationVisualizer(sim_horst.params)

# cutoff_pct = 0.05


# for group_name, experiment_ids in experiment_groups_horst.items():
    
#     print(f"\n{'='*40}")
#     print(f"Processing Group: {group_name}")
#     print(f"IDs: {experiment_ids}")
#     print(f"{'='*40}")

#     try:
#         # Run analysis
#         exp_data = analyzer_horst.analyze(
#             exp_ids=experiment_ids, 
#             data_path=path_exp_horst, 
#             cutoff_pct=cutoff_pct, 
#             time_step=sim_horst.params.time_step
#         )

#         # Run Simulation
#         states_history, inputs_history = sim_horst.run_validation(exp_data)

#         # Plot 
#         viz_horst.plot_comparison(states_history, inputs_history, experiment_ids, path_exp_horst, cutoff_pct, save_fig=True, group_name=group_name, experiment_name="OptiHorst")
#         # print(viz_horst.get_relative_error_table(states_history, inputs_history, experiment_ids, path_exp_horst, cutoff_pct, experiment_name="OptiHorst"))
#         # viz_horst.plot_layer_temperatures(states_history)
#         # viz_horst.plot_heat_transfer_coefficients(states_history)
#         # viz_horst.plot_detailed(states_history, inputs_history, experiment_ids)
#         # viz_horst.plot_roughness(states_history, experiment_ids[0])
#         # viz_horst.plot_frost_distribution(states_history)
#         # viz_horst.plot_debug(states_history, inputs_history, experiment_ids[0])
        

#     except Exception as e:
#         import traceback
#         print(f"!!! CRITICAL FAILURE in Group {group_name} !!!")
#         print(f"Error Message: {e}")
#         traceback.print_exc()



<!-- ### Run Mutliple Simulations -->

### Parameter Optimierung

In [ ]:
from pathlib import Path
import sys
import os
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# pip install scikit-optimize matplotlib
from skopt import gp_minimize
from skopt.space import Integer, Categorical
from skopt.utils import use_named_args
from skopt.plots import plot_convergence

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Assuming your updated class is in this module
from frost_evaporator import MultiExperimentAnalyzer, FrostEvaporatorSimulation, SimulationVisualizer

###########################################################################################
# Initialisation
###########################################################################################

# 1. Define Discrete Search Space (Steps of 0.05)
space = [
    # --- Numeric Factors (Mapped from Integer) ---
    Integer(14, 26, name='h_conv_air_idx'),         # 0.6 - 1.4
    Integer(14, 26, name='h_conv_ref_2ph_idx'),     # 0.6 - 1.4
    Integer(10, 26, name='betta_air_idx'),          # 0.6 - 1.4
    Integer(14, 26, name='surface_density_idx'),    # 0.6 - 1.4
    Integer(14, 26, name='k_frost_idx'),            # 0.6 - 1.4
    Integer(14, 26, name='frost_diffusion_idx'),

    # --- Categorical Choices (Model Selection) ---
    Categorical(['jonas_diss', 'D8', 'da_silva_paper', 'wang_2012'], name='frost_density_choice'),
    # Categorical(['A', 'da_silva_paper'],                name='frost_conductivity_choice'),
    Categorical(['oneal_tree_1984', 'lee_1997', 'yonko_sepsy_1967', 'maxwell_eucken'],              name='frost_conductivity_choice'),
    Categorical(['Wang', 'VDI'],                        name='h_conv_air_choice'),
]


# 2. Setup Simulation Objects
cutoff_pct = 0.05

# --- Abt Setup ---
analyzer_abt = MultiExperimentAnalyzer(experiment_type='OptiAbt')
sim_abt = FrostEvaporatorSimulation("config_OptiAbt.yaml")
viz_abt = SimulationVisualizer(sim_abt.params)
path_exp_abt = Path(r"D:\mbc_nba\OptiAbt_Daten\Versuche\CSV_Export")
experiment_groups_abt = {
    "Abt-a": [56],
    "Abt-b": [40],
    "Abt-c": [28],
    "Abt-d": [13],
}

# --- Horst Setup ---
analyzer_horst = MultiExperimentAnalyzer(experiment_type='OptiHorst')
sim_horst      = FrostEvaporatorSimulation("config_OptiHorst.yaml")
viz_horst      = SimulationVisualizer(sim_horst.params)
path_exp_horst = Path(r"D:\mbc_nba\OptiHorst\Daten\Abtauen\CSV_Export")
# experiment_groups_horst = {
#     "Horst-a": [96],
#     "Horst-b": [25],
#     "Horst-c": [45],
#     "Horst-d": [102],
#     "Horst-e": [60],
# }

# 3. Pre-load Data (Do this once to save time)
print("Pre-loading experiment data...")
cached_abt_data = {}
for name, ids in experiment_groups_abt.items():
    cached_abt_data[name] = analyzer_abt.analyze(exp_ids=ids, data_path=path_exp_abt, cutoff_pct=cutoff_pct, time_step=sim_abt.params.time_step)

# cached_horst_data = {}
# for name, ids in experiment_groups_horst.items():
#     cached_horst_data[name] = analyzer_horst.analyze(exp_ids=ids, data_path=path_exp_horst, cutoff_pct=cutoff_pct, time_step=sim_horst.params.time_step)


###########################################################################################
# The Objective Function
###########################################################################################

@use_named_args(space)
def objective_function(**kwargs):
    start_time = time.time()
    
    # 1. Convert Optimizer Integers back to Float Factors
    new_factors = {
        'h_conv_air':          kwargs['h_conv_air_idx'] * 0.05,
        'h_conv_ref_2ph':      kwargs['h_conv_ref_2ph_idx'] * 0.05,
        'betta_air':           kwargs['betta_air_idx'] * 0.05,
        'surface_density':     kwargs['surface_density_idx'] * 0.05,
        'k_frost':             kwargs['k_frost_idx'] * 0.05,
        'frost_diffusion':     kwargs['frost_diffusion_idx'] * 0.05,
    }

    # 2. Extract Categorical Choices
    new_model_choices = {
        'frost_density_choice':      kwargs['frost_density_choice'],
        'frost_conductivity_choice': kwargs['frost_conductivity_choice'],
        'h_conv_air_choice':         kwargs['h_conv_air_choice'],
    }

    # Print nicely formatted params
    print(f"\n--- New Iteration ---")
    print("Factors:", {k: round(v, 2) for k, v in new_factors.items()})
    print("Models: ", new_model_choices)

    total_summed_error = 0.0

    abt_weights = {
        "Abt-a": 1.0,
        "Abt-b": 1.0,
        "Abt-c": 1.0,
        "Abt-d": 0.25
    }

    horst_weights = {
        "Horst-a": 1.0,
        "Horst-b": 1.0,
        "Horst-c": 1.0,
        "Horst-d": 1.0,
        "Horst-e": 0.25,
    }

    # --- Run Abt Simulations ---
    try:
        sim_abt.update_correction_factors(new_factors, new_model_choices)
        
        for name, exp_data in cached_abt_data.items():
            states, inputs = sim_abt.run_validation(exp_data)
            exp_id = experiment_groups_abt[name][0]
            
            df_err = viz_abt.get_relative_error_table(
                states, inputs, [exp_id], path_exp_abt, cutoff_pct, experiment_name="OptiAbt"
            )
            
            row = df_err.loc[exp_id].fillna(0.0)

            # 1. Extract individual relative errors (for cleaner printing and math)
            if name == "Abt-a" or name == "Abt-b":
                err_dp = (row["dp"]) ** 2
            else:
                err_dp = 0

            err_q_base = (row["Q"])**2

            err_mfrost = (row["m_frost"])**2

            # 2. Calculate the base error for this run
            single_run_error = err_dp + err_q_base + err_mfrost
            
            # 3. Apply the Importance Weight
            weight = abt_weights.get(name, 1.0)
            weighted_error = single_run_error * weight
            
            total_summed_error += weighted_error
            
            # 4. Print the comprehensive breakdown
            print(f"   -> {name} | Base Error: {single_run_error:.1f} (Exp Weight: {weight} -> Final: {weighted_error:.1f})")
            print(f"      Components: dp={err_dp:.1f} | Q={err_q_base:.1f} | m_frost={err_mfrost:.1f}")

    except Exception as e:
        print(f"!!! Error in Abt Simulation: {e}")
        return 9999.0 # Return huge penalty

    # # --- Run Horst Simulations ---
    # try:
    #     sim_horst.update_correction_factors(new_factors, new_model_choices)
        
    #     for name, exp_data in cached_horst_data.items():
    #         # Run Simulation
    #         states, inputs = sim_horst.run_validation(exp_data)
            
    #         # --- NEW ERROR CALCULATION ---
    #         exp_id = experiment_groups_horst[name][0]
            
    #         df_err = viz_horst.get_relative_error_table(
    #             states, inputs, [exp_id], path_exp_horst, cutoff_pct, experiment_name="OptiHorst"
    #         )
            
    #         row = df_err.loc[exp_id].fillna(0.0)
            
    #         # 1. Extract individual relative errors (for cleaner printing and math)
    #         err_dp = (row["dp"]) ** 2
    #         err_q_base = (row["Q"])**2
    #         err_mfrost = (row["m_frost"])**2

    #         # 2. Calculate the base error for this run
    #         single_run_error = err_dp + err_q_base + err_mfrost
            
    #         # 3. Apply the Importance Weight
    #         weight = horst_weights.get(name, 1.0)
    #         weighted_error = single_run_error * weight
            
    #         total_summed_error += weighted_error
            
    #         # 4. Print the comprehensive breakdown
    #         print(f"   -> {name} | Base Error: {single_run_error:.1f} (Exp Weight: {weight} -> Final: {weighted_error:.1f})")
    #         print(f"      Components: dp={err_dp:.1f} | Q={err_q_base:.1f} | m_frost={err_mfrost:.1f}")
    # except Exception as e:
    #     print(f"!!! Error in Horst Simulation: {e}")
    #     return 99999.0

    duration = time.time() - start_time
    print(f"=== Total Error: {total_summed_error:.4f} (Time: {duration:.1f}s) ===")
    
    return total_summed_error


###########################################################################################
# Run Optimization
###########################################################################################

print("\nStarting Bayesian Optimization...")
print("The optimizer will intelligently explore the ranges to minimize total error.")

# gp_minimize uses Gaussian Processes (Bayesian Optimization)
result = gp_minimize(
    objective_function,
    space,
    n_calls=300,
    n_initial_points=100,
    random_state=42,
    n_jobs=1
)

###########################################################################################
# Results & Visualization
###########################################################################################

print("\n" + "="*50)
print("OPTIMIZATION FINISHED")
print("="*50)
print(f"Best Total Error: {result.fun:.4f}")

# Map results back to names
best_results_dict = dict(zip([d.name for d in space], result.x))

print("\nBest Configuration:")

# 1. Print Factors
print("--- Factors ---")
for k, v in best_results_dict.items():
    if "_idx" in k:
        real_name = k.replace("_idx", "")
        real_val = v * 0.05
        print(f"  {real_name}: {real_val:.2f}")

# 2. Print Choices
print("--- Model Choices ---")
for k, v in best_results_dict.items():
    if "_choice" in k:
        print(f"  {k}: {v}")

# --- Generate and Save Convergence Plot ---
try:
    print("\nGenerating convergence plot...")
    plt.figure(figsize=(10, 6))
    
    # Plot erstellen
    plot_convergence(result)
    
    plt.title("Optimization Convergence (Mixed Integer/Categorical)")
    plt.ylabel("Min Total Error Found")
    plt.xlabel("Number of Iterations")
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    # SPEICHERN statt Anzeigen
    filename = "optimization_result.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Plot saved successfully as '{filename}'")
    
    # Speicher freigeben (wichtig bei Loops oder Server-Betrieb)
    plt.close()
    
except Exception as e:
    print(f"Could not generate plot: {e}")